In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

**charger les données**

In [ ]:
iris = load_iris(as_frame=True)
X = iris.data            # 4 mesures : longueur/largeur des sépales et des pétales
y = iris.target          # 0 = setosa, 1 = versicolor, 2 = virginica
noms_classes = ["setosa", "versicolor", "virginica"]

print(X.shape[0], "fleurs,", X.shape[1], "mesures")
X.head()

**Séparation données**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42   # stratify : même proportion d'espèces partout
)
print(f"Entraînement : {len(X_train)}  |  Test : {len(X_test)}")

**Softmax function**

In [ ]:
modele_softmax = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=1000)),    # softmax automatique avec 3 classes
])
modele_softmax.fit(X_train, y_train)
y_pred_softmax = modele_softmax.predict(X_test)
print(f"Accuracy (softmax) : {accuracy_score(y_test, y_pred_softmax):.3f}")

**One-Vs-Rest**

In [ ]:
modele_ovr = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", OneVsRestClassifier(LogisticRegression(max_iter=1000))),   # 3 modèles binaires
])
modele_ovr.fit(X_train, y_train)
y_pred_ovr = modele_ovr.predict(X_test)
print(f"Accuracy (One-vs-Rest) : {accuracy_score(y_test, y_pred_ovr):.3f}")

**comparaison de deux méthodes**

In [ ]:
fleur = X_test.iloc[[0]]
print(f"Vraie espèce : {noms_classes[y_test.iloc[0]]}\n")

proba_softmax = modele_softmax.predict_proba(fleur)[0]
proba_ovr = modele_ovr.predict_proba(fleur)[0]

pd.DataFrame({
    "classe": noms_classes,
    "softmax": proba_softmax.round(3),
    "One-vs-Rest": proba_ovr.round(3),
})

**vérification sans sklearn**

In [ ]:
scores = modele_softmax.decision_function(fleur)[0]   # scores bruts z (un par classe)
print("Scores bruts z :", scores.round(2))

exp_z = np.exp(scores)
print("e^z            :", exp_z.round(2))

proba_main = exp_z / exp_z.sum()                      # formule du softmax
print("Softmax (main) :", proba_main.round(3))
print("scikit-learn   :", proba_softmax.round(3))     # identique !